# Enron Cleaned

This is the notebook used to preprocess the Enron Cleaned dataset which is a deduplicated version of the Enron dataset provided by Andrea on 2026-02-27.

In [122]:
import sys
import re

import pandas as pd

from from_root import from_root
from pathlib import Path

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_txt, read_rds, read_jsonl, write_jsonl, write_rds

In [123]:
base_loc = Path("/Volumes/BCross/datasets")

enron_base_loc = base_loc / "enron_cleaned"

test_base_loc = base_loc / "author_verification/test"
training_base_loc = base_loc / "author_verification/training"

test_save_dir = test_base_loc / "Enron Cleaned"
test_save_dir.mkdir(parents=True, exist_ok=True)

training_save_dir = training_base_loc / "Enron Cleaned"
training_save_dir.mkdir(parents=True, exist_ok=True)

## Read in the previous metadata

Read in the previous metadata, check the colnames for processing and look at distinct corpora.

In [124]:
test_metadata = read_rds(test_base_loc / "metadata.rds")
training_metadata = read_rds(training_base_loc / "metadata.rds")

print(f"Columns in the metadata: {test_metadata.columns.tolist()}")

Columns in the metadata: ['problem', 'corpus', 'known_author', 'unknown_author']


In [125]:
sorted(test_metadata['corpus'].drop_duplicates().tolist())

['ACL',
 'All-the-news',
 'Amazon',
 'Enron',
 'Enron Cleaned',
 'IMDB',
 "Koppel's Blogs",
 'Perverted Justice',
 'Reddit',
 'StackExchange',
 'The Apricity',
 'The Telegraph',
 'TripAdvisor',
 'Wiki',
 'Yelp']

In [126]:
test_metadata[test_metadata['corpus'] == "Enron Cleaned"].head()

,problem,corpus,known_author,unknown_author
12434,Kevin.hyatt vs Kevin.hyatt,Enron Cleaned,Kevin.hyatt,Kevin.hyatt
12435,Kevin.hyatt vs Kimberly.watson,Enron Cleaned,Kevin.hyatt,Kimberly.watson
12436,Kimberly.watson vs Kimberly.watson,Enron Cleaned,Kimberly.watson,Kimberly.watson
12437,Kimberly.watson vs Larry.campbell,Enron Cleaned,Kimberly.watson,Larry.campbell
12438,Larry.campbell vs Larry.campbell,Enron Cleaned,Larry.campbell,Larry.campbell


## Read in previous known to view layout

Again used to check out the columns of the previous Enron data.

In [127]:
known = read_jsonl(test_base_loc / "Enron/known_raw.jsonl")

print(f"Columns in the metadata: {known.columns.tolist()}")

Columns in the metadata: ['doc_id', 'text', 'corpus', 'author', 'texttype']


In [128]:
known.head()

,doc_id,text,corpus,author,texttype
0,known [Kevin.hyatt - Mail_1].txt,school supplies sports soccer wants to be teac...,Enron,Kevin.hyatt,known
1,known [Kevin.hyatt - Mail_3].txt,"I've installed a new one, but the only windows...",Enron,Kevin.hyatt,known
2,known [Kevin.hyatt - Mail_4].txt,This was conducive towards the creation of a t...,Enron,Kevin.hyatt,known
3,known [Kevin.hyatt - Mail_5].txt,Kissing the bride for more than 5 seconds may ...,Enron,Kevin.hyatt,known
4,known [Kimberly.watson - Mail_1].txt,"Yes, we are already putting a list together an...",Enron,Kimberly.watson,known


## Helper Functions

These are used to read in files and process names of files/folders to return valuable metadata from them.

In [129]:
def return_problem_folder_names(file_dir):
    """Returns the names of the folders within the main directory i.e. problem names"""
    
    folders = [p for p in file_dir.iterdir() if p.is_dir()]
    
    filenames = sorted([p.name for p in folders])
    
    return filenames

In [130]:
def get_problem_metadata(problem):
    """Returns (known_author, unknown_author) from "Known vs Unknown"""
    known, unknown = problem.split(" vs ", 1)
    return known.strip(), unknown.strip()

In [131]:
def get_txt_files_in_dir(file_dir):
    
    txt_files = sorted([p.name for p in file_dir.iterdir() if p.is_file() and p.suffix.lower() == ".txt"])
    return txt_files

In [132]:
def get_file_metadata(filename):
    """
    Returns (status, author)
    e.g. "known [Kevin.hyatt - Mail_1].txt" -> ("known", "Kevin.hyatt")
    """
    
    PAT = re.compile(r'^(known|unknown)\s+\[(.*?)\s*-\s*.*\]\.txt$', re.IGNORECASE)

    m = PAT.match(filename.strip())
    if not m:
        raise ValueError(f"Unrecognized format: {filename!r}")
    status = m.group(1).lower()
    author = m.group(2).strip()
    return status, author

## Process Files

This is the function that reads all of the files and creates the metadata and the actual known and unknown files.

In [133]:
def process_files(file_dir):
    """Process the files in a directory and output metadata, known and unknown dataframes"""
    
    # Create empty lists for both data types
    metadata_list = []
    document_list = []
    
    # This gets all of the file directories of problems
    problems_in_folder = return_problem_folder_names(file_dir)
    
    for problem in problems_in_folder:
        
        # Pulls authors from folder name like Kevin.hyatt vs Kevin.hyatt
        known_author, unknown_author = get_problem_metadata(problem)
        
        problem_dir = file_dir / problem
        
        # Returns all of the .txt files in a directory
        problem_files = get_txt_files_in_dir(problem_dir)
        
        # Create the metadata row since we have all of this info
        metadata_row = {
            "problem": problem,
            "corpus": "Enron Cleaned",
            "known_author": known_author,
            "unknown_author": unknown_author
        }
            
        metadata_list.append(metadata_row)
        
        # Now we need to read in the data
        for file_ in problem_files:
            
            # Gets the text type and the author from filename like known [Kevin.hyatt - Mail_1].txt
            status, author = get_file_metadata(file_)
            
            file_loc = problem_dir / file_
            
            # Read in the text
            doc_text = read_txt(file_loc)
            
            # Create the row for the document data and append
            document_row = {
                "doc_id": file_,
                "text": doc_text,
                "corpus": "Enron Cleaned",
                "author": author,
                "texttype": status
            }
            
            document_list.append(document_row)
            
    # Combine the metadata rows
    metadata = pd.DataFrame(metadata_list)
    
    # Combine the document rows
    document_df = pd.DataFrame(document_list)
    
    # Filter into known and unknown and drop duplicates since may be duplicated across problems
    known = document_df[document_df['texttype'] == 'known'].drop_duplicates()
    unknown = document_df[document_df['texttype'] == 'unknown'].drop_duplicates()
    
    return metadata, known, unknown

## Run the code and save

Here we run the code and save the files, the known and unknown can be saved as the are but we first filter out 'Enron Cleaned' from the original metadata, then union the original and the new metadata together before saving.

### Test

In [134]:
test_base_dir = enron_base_loc / "test"
test_metadata_new, test_known, test_unknown = process_files(test_base_dir)

In [135]:
write_jsonl(test_known, test_save_dir / "known_raw.jsonl")
write_jsonl(test_unknown, test_save_dir / "unknown_raw.jsonl")

In [136]:
test_metadata = test_metadata[test_metadata['corpus'] != "Enron Cleaned"]

test_metadata_combined = pd.concat([test_metadata, test_metadata_new], ignore_index=True)

write_rds(test_metadata_combined, test_base_loc / "metadata.rds")

### Training

In [137]:
training_base_dir = enron_base_loc / "training"
training_metadata_new, training_known, training_unknown = process_files(training_base_dir)

In [138]:
write_jsonl(training_known, training_save_dir / "known_raw.jsonl")
write_jsonl(training_unknown, training_save_dir / "unknown_raw.jsonl")

In [139]:
training_metadata = training_metadata[training_metadata['corpus'] != "Enron Cleaned"]

training_metadata_combined = pd.concat([training_metadata, training_metadata_new], ignore_index=True)

write_rds(training_metadata_combined, training_base_loc / "metadata.rds")